# FreshCart ELT on Google Colab: APIs → Snowflake → SQL

This notebook runs the full pipeline from Colab:

1. **Extract**: Python calls the APIs and saves the raw JSON.
2. **Load**: the raw files are uploaded to Snowflake's **bronze** tables (`PUT` + `COPY INTO`).
3. **Transform**: SQL inside Snowflake cleans, deduplicates and enriches into **silver**, then publishes **gold**.

**Before you start**, add your Snowflake login in Colab's **Secrets** panel (the 🔑 key icon in the left sidebar). Switch on *Notebook access* for each one.

| Secret name | Example | Needed? |
|---|---|---|
| `SNOWFLAKE_ACCOUNT` | `myorg-myaccount` (Snowsight → your name → Account → Copy account identifier) | yes |
| `SNOWFLAKE_USER` | `CHANDRA` | yes |
| `SNOWFLAKE_PASSWORD` | your password, **or** a programmatic access token | one of these two |
| `SNOWFLAKE_PRIVATE_KEY` | the full text of an `rsa_key.p8` key-pair file | one of these two |
| `SNOWFLAKE_ROLE` | `SYSADMIN` (default) | optional |

Colab can't open a browser login window, so SSO (`externalbrowser`) does not work here. If your account requires MFA and blocks plain password logins, use a key pair or a programmatic access token instead.

Run the cells from top to bottom.

In [1]:
!apt-get update -qq
!apt-get install -y rabbitmq-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 logrotate
Suggested packages:
  erlang erlang-manpages erlang-doc lksctp-tools bsd-mailx | mailx
The following NEW packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 lo

In [2]:
!service rabbitmq-server start


[ OK ]


In [3]:
!rabbitmqctl status

Status of node rabbit@7f7f19fe2ec1 ...
Runtime

OS PID: 3093
OS: Linux
Uptime (seconds): 20
Is under maintenance?: false
RabbitMQ version: 3.12.1
RabbitMQ release series support status: supported
Node name: rabbit@7f7f19fe2ec1
Erlang configuration: Erlang/OTP 25 [erts-13.2.2.5] [source] [64-bit] [smp:2:2] [ds:2:2:10] [async-threads:1] [jit:ns]
Crypto library: OpenSSL 3.0.13 30 Jan 2024
Erlang processes: 272 used, 1048576 limit
Scheduler run queue: 1
Cluster heartbeat timeout (net_ticktime): 60

Plugins

Enabled plugin file: /etc/rabbitmq/enabled_plugins
Enabled plugins:


Data directory

Node data directory: /var/lib/rabbitmq/mnesia/rabbit@7f7f19fe2ec1
Raft data directory: /var/lib/rabbitmq/mnesia/rabbit@7f7f19fe2ec1/quorum/rabbit@7f7f19fe2ec1

Config files


Log file(s)

 * /var/log/rabbitmq/rabbit@7f7f19fe2ec1.log
 * <stdout>

Alarms

(none)

Memory

Total memory used: 0.1303 gb
Calculation strategy: rss
Memory high watermark setting: 0.4 of available memory, computed to: 5.4423 gb



In [4]:
!pip install pika


In [5]:
%%writefile dispatcher.py
import pika
import json
import random
import time
import uuid
from datetime import datetime

connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)
channel = connection.channel()

channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

cities = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "whitefield"]
}

try:
    while True:
        city = random.choice(list(cities.keys()))
        zone = random.choice(cities[city])
        request_type = random.choice(["ride", "food"])

        routing_key = f"{city}.{zone}.{request_type}"

        message = {
            "id": str(uuid.uuid4()),
            "city": city,
            "zone": zone,
            "type": request_type,
            "ts": datetime.now().isoformat()
        }

        channel.basic_publish(
            exchange="rapido-dispatch",
            routing_key=routing_key,
            body=json.dumps(message),
            properties=pika.BasicProperties(delivery_mode=2)
        )

        print("Published:", routing_key, message, flush=True)
        time.sleep(2)

except KeyboardInterrupt:
    print("Dispatcher stopped.")

finally:
    connection.close()

Writing dispatcher.py


In [6]:
%%writefile driver.py
import pika
import sys
import json

if len(sys.argv) != 3:
    print("Usage: python driver.py NAME PATTERN")
    sys.exit(1)

name = sys.argv[1]
pattern = sys.argv[2]

connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)
channel = connection.channel()

channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

result = channel.queue_declare(
    queue="",
    exclusive=True,
    auto_delete=True
)

queue_name = result.method.queue

channel.queue_bind(
    exchange="rapido-dispatch",
    queue=queue_name,
    routing_key=pattern
)

print(f"{name} is waiting for: {pattern}", flush=True)

def callback(ch, method, properties, body):
    message = json.loads(body)
    print(f"\nDriver: {name}")
    print(f"Routing key: {method.routing_key}")
    print(f"Message: {message}", flush=True)
    ch.basic_ack(delivery_tag=method.delivery_tag)

channel.basic_consume(
    queue=queue_name,
    on_message_callback=callback,
    auto_ack=False
)

try:
    channel.start_consuming()
except KeyboardInterrupt:
    print(f"{name} stopped.")
finally:
    connection.close()

Writing driver.py


In [8]:
!python -u driver.py Ravi 'bangalore.koramangala.*' > ravi.log 2>&1 &
!python -u driver.py Meena 'bangalore.*.food' > meena.log 2>&1 &
!python -u driver.py Control 'bangalore.#' > control.log 2>&1 &

In [9]:
!sleep 3
!cat ravi.log
!cat meena.log
!cat control.log

Ravi is waiting for: bangalore.koramangala.*
Meena is waiting for: bangalore.*.food
Control is waiting for: bangalore.#


In [10]:
!timeout 120s python -u dispatcher.py > dispatcher.log 2>&1

In [11]:
!cat dispatcher.log

Published: chennai.whitefield.ride {'id': 'fd0867a2-36f0-4587-bb94-1eea13d8bf3c', 'city': 'chennai', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-13T00:10:20.327306'}
Published: bangalore.koramangala.food {'id': 'b73d4c32-e29d-4da0-9f2f-e84ba12d52bb', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:22.327797'}
Published: bangalore.indiranagar.food {'id': '31a3bc98-9715-4e52-86ce-c9027f3743d1', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'food', 'ts': '2026-11-13T00:10:24.328439'}
Published: bangalore.koramangala.food {'id': '2e3014e5-1a26-4e71-b8b0-dc72aeb4d03c', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:26.329005'}
Published: bangalore.koramangala.food {'id': 'aeeab85b-2b6f-4be1-a30d-b4f8aec22ab7', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:28.329594'}
Published: bangalore.indiranagar.food {'id': 'bf426ee3-cc81-4819-b797-58b84992346a', 'city': 'bangalo

In [12]:
!echo "----- RAVI -----"
!cat ravi.log
!echo "----- MEENA -----"
!cat meena.log
!echo "----- CONTROL -----"
!cat control.log

----- RAVI -----
Ravi is waiting for: bangalore.koramangala.*

Driver: Ravi
Routing key: bangalore.koramangala.food
Message: {'id': 'b73d4c32-e29d-4da0-9f2f-e84ba12d52bb', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:22.327797'}

Driver: Ravi
Routing key: bangalore.koramangala.food
Message: {'id': '2e3014e5-1a26-4e71-b8b0-dc72aeb4d03c', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:26.329005'}

Driver: Ravi
Routing key: bangalore.koramangala.food
Message: {'id': 'aeeab85b-2b6f-4be1-a30d-b4f8aec22ab7', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T00:10:28.329594'}

Driver: Ravi
Routing key: bangalore.koramangala.ride
Message: {'id': 'b7d97798-8641-4106-beac-2da6b0a55516', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-13T00:10:40.333127'}

Driver: Ravi
Routing key: bangalore.koramangala.ride
Message: {'id': '2ea45a08-efa8-42f1-acc7-35f5fd08e336', 'cit

In [13]:
!echo "Ravi:"
!grep -c "Routing key: bangalore.koramangala.food" ravi.log

!echo "Meena:"
!grep -c "Routing key: bangalore.koramangala.food" meena.log

!echo "Control:"
!grep -c "Routing key: bangalore.koramangala.food" control.log

Ravi:
8
Meena:
8
Control:
8


In [14]:
%%writefile driver.py
import pika
import sys
import json

if len(sys.argv) != 3:
    print("Usage: python driver.py NAME PATTERN")
    sys.exit(1)

name = sys.argv[1]
pattern = sys.argv[2]

connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)
channel = connection.channel()

channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

result = channel.queue_declare(queue="", exclusive=True, auto_delete=True)
queue_name = result.method.queue

channel.queue_bind(
    exchange="rapido-dispatch",
    queue=queue_name,
    routing_key=pattern
)

print(f"{name} is waiting for: {pattern}", flush=True)

def callback(ch, method, properties, body):
    message = json.loads(body)

    if "zone" not in message:
        print(f"REJECTED: {method.routing_key} | Missing zone | NACK", flush=True)
        ch.basic_nack(
            delivery_tag=method.delivery_tag,
            requeue=False
        )
    else:
        print(f"PROCESSED: {method.routing_key} | {message}", flush=True)
        ch.basic_ack(delivery_tag=method.delivery_tag)

channel.basic_consume(
    queue=queue_name,
    on_message_callback=callback,
    auto_ack=False
)

try:
    channel.start_consuming()
except KeyboardInterrupt:
    print(f"{name} stopped.")
    channel.stop_consuming()
finally:
    connection.close()

Overwriting driver.py


In [15]:
%%writefile dispatcher.py
import pika
import json
import random
import time
import uuid
from datetime import datetime

connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)
channel = connection.channel()

channel.exchange_declare(
    exchange="rapido-dispatch",
    exchange_type="topic",
    durable=True
)

cities = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "whitefield"]
}

count = 0

try:
    while True:
        count += 1
        city = random.choice(list(cities.keys()))
        zone = random.choice(cities[city])
        request_type = random.choice(["ride", "food"])

        routing_key = f"{city}.{zone}.{request_type}"

        message = {
            "id": str(uuid.uuid4()),
            "city": city,
            "type": request_type,
            "ts": datetime.now().isoformat()
        }

        if count % 10 != 0:
            message["zone"] = zone

        channel.basic_publish(
            exchange="rapido-dispatch",
            routing_key=routing_key,
            body=json.dumps(message),
            properties=pika.BasicProperties(delivery_mode=2)
        )

        print(f"Published #{count}: {routing_key} | {message}", flush=True)
        time.sleep(2)

except KeyboardInterrupt:
    print("Dispatcher stopped.")
finally:
    connection.close()

Overwriting dispatcher.py


In [17]:
!python -u driver.py Ravi 'bangalore.koramangala.*' > ravi_d3.log 2>&1 &
!python -u driver.py Meena 'bangalore.*.food' > meena_d3.log 2>&1 &
!python -u driver.py Control 'bangalore.#' > control_d3.log 2>&1 &

In [18]:
!sleep 3
!cat ravi_d3.log
!cat meena_d3.log
!cat control_d3.log

Ravi is waiting for: bangalore.koramangala.*
Meena is waiting for: bangalore.*.food
Control is waiting for: bangalore.#


In [19]:
!timeout 120s python -u dispatcher.py > dispatcher_d3.log 2>&1

In [20]:
!cat dispatcher_d3.log

Published #1: chennai.tnagar.ride | {'id': '6507e449-6352-4d27-b4b5-167d0efff565', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T00:52:31.930464', 'zone': 'tnagar'}
Published #2: bangalore.indiranagar.food | {'id': 'e9818600-c864-4572-a407-30bad7c2ad73', 'city': 'bangalore', 'type': 'food', 'ts': '2026-11-13T00:52:33.930998', 'zone': 'indiranagar'}
Published #3: chennai.whitefield.ride | {'id': '6ce3e454-3118-4d17-a732-7745a35c7524', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T00:52:35.931636', 'zone': 'whitefield'}
Published #4: bangalore.koramangala.food | {'id': '3a1adf77-f2bb-4494-866b-374b802e57de', 'city': 'bangalore', 'type': 'food', 'ts': '2026-11-13T00:52:37.932201', 'zone': 'koramangala'}
Published #5: chennai.whitefield.ride | {'id': 'f80cdc84-6592-4a6d-a138-0d7eea4d8fe0', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T00:52:39.933527', 'zone': 'whitefield'}
Published #6: bangalore.indiranagar.food | {'id': 'c7fddb06-912e-42e5-ba89-ee9b8010c8d6', 'city'

In [21]:
!echo "------ RAVI ------"
!cat ravi_d3.log

!echo "------ MEENA ------"
!cat meena_d3.log

!echo "------ CONTROL ------"
!cat control_d3.log

------ RAVI ------
Ravi is waiting for: bangalore.koramangala.*
PROCESSED: bangalore.koramangala.food | {'id': '3a1adf77-f2bb-4494-866b-374b802e57de', 'city': 'bangalore', 'type': 'food', 'ts': '2026-11-13T00:52:37.932201', 'zone': 'koramangala'}
PROCESSED: bangalore.koramangala.ride | {'id': 'dedfdfef-7c17-494c-be5e-e6113303ec24', 'city': 'bangalore', 'type': 'ride', 'ts': '2026-11-13T00:52:55.938263', 'zone': 'koramangala'}
PROCESSED: bangalore.koramangala.food | {'id': '42f18d7e-143b-4a31-97d3-94c6901ae71f', 'city': 'bangalore', 'type': 'food', 'ts': '2026-11-13T00:52:59.939609', 'zone': 'koramangala'}
PROCESSED: bangalore.koramangala.ride | {'id': '682a250f-d965-4de6-a54b-315cbd320a7f', 'city': 'bangalore', 'type': 'ride', 'ts': '2026-11-13T00:53:05.941581', 'zone': 'koramangala'}
PROCESSED: bangalore.koramangala.ride | {'id': '171ac00e-e62d-4442-9035-d721514f6d5a', 'city': 'bangalore', 'type': 'ride', 'ts': '2026-11-13T00:53:17.945122', 'zone': 'koramangala'}
PROCESSED: bangalore.

In [22]:
!pkill -f '[d]river.py' || true

## 1. Install libraries

In [ ]:
!pip install -q snowflake-connector-python requests python-dotenv

## 2. Create the project files
Each cell below writes one file into `/content/freshcart_elt`. Just run them; you can read them later.

In [ ]:
!mkdir -p /content/freshcart_elt/sql

In [ ]:
%%writefile /content/freshcart_elt/requirements.txt
snowflake-connector-python>=3.6
requests>=2.31
python-dotenv>=1.0

Overwriting /content/freshcart_elt/requirements.txt


In [ ]:
%%writefile /content/freshcart_elt/mock_data.py
"""Mock API responses: the 8 FreshCart orders from the course, split by the channel
they came from, plus CRM, FX and pincode responses. Used with --mock so the whole
pipeline runs end to end without any real API.

Note the two order APIs use DIFFERENT JSON shapes. That is normal in real life,
and it is why bronze stores raw JSON and silver unifies it with SQL.
"""

# Web export: flat JSON (rows 1, 3, 4, 5, 6, 8 of the course table)
WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# Mobile API: nested JSON, camelCase, day-first dates (rows 2 and 7)
MOBILE_ORDERS = [
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# CRM: a log of tier changes. The SQL turns this into an SCD Type 2 table.
CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# Same shape as https://api.frankfurter.app/2026-09-01..2026-09-03?from=USD&to=INR
FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# Same shape as https://api.postalpincode.in/pincode/600041
_PIN = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),
        "400050": ("Mumbai", "Maharashtra")}


def pincode_response(pin):
    if pin not in _PIN:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = _PIN[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

Overwriting /content/freshcart_elt/mock_data.py


In [ ]:
%%writefile /content/freshcart_elt/extract.py
"""EXTRACT: call each API and save the responses unchanged as gzipped newline-delimited
JSON files in the local landing folder.

No cleaning happens here (that is the point of ELT). Each record is wrapped in a
small envelope so we always know where and when it came from:

    {"source": "web", "run_id": "...", "extracted_at": "...", "data": {<original record>}}
"""
import gzip
import json
import logging
import os
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import mock_data

log = logging.getLogger("extract")


# ---------------------------------------------------------------- HTTP helpers
def http_session():
    """A requests session that retries transient failures with exponential backoff
    (1 s, 2 s, 4 s, ...) and honours the server's Retry-After header on 429."""
    retry = Retry(total=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",), respect_retry_after_header=True)
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.headers["User-Agent"] = "freshcart-elt/1.0"
    return s


def _auth(token_env):
    token = os.getenv(token_env)
    return {"Authorization": f"Bearer {token}"} if token else {}


def paginate(session, url, params=None, headers=None, items_key="data", next_key="next",
             timeout=30, max_pages=10_000):
    """Yield records from an API that returns {"data": [...], "next": "<url or null>"}.
    Change items_key / next_key to match your API (e.g. "results" / "next_page_url").
    If your API uses page numbers instead, loop on params["page"] until a page is empty."""
    page_url, page_params = url, dict(params or {})
    for _ in range(max_pages):
        r = session.get(page_url, params=page_params, headers=headers, timeout=timeout)
        r.raise_for_status()
        body = r.json()
        items = body.get(items_key, []) if isinstance(body, dict) else body
        yield from items
        nxt = body.get(next_key) if isinstance(body, dict) else None
        if not nxt:
            return
        page_url, page_params = nxt, None      # the next link already carries the query
    raise RuntimeError(f"{url}: more than {max_pages} pages, stopping")


# ---------------------------------------------------------------- landing files
def write_landing(source, records, run_id, run_date):
    """Write records as one .json.gz file per source per run. Returns the path, or None."""
    records = list(records)
    if not records:
        log.info("%s: 0 records, nothing written", source)
        return None
    folder = Path(os.getenv("LANDING_DIR", "landing")) / source / run_date.isoformat()
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps({"source": source, "run_id": run_id, "extracted_at": now,
                                "data": rec}, ensure_ascii=False) + "\n")
    log.info("%s: %d records -> %s", source, len(records), path)
    return path


# ---------------------------------------------------------------- one function per API
def web_orders(session, run_date, mock=False):
    if mock:
        return mock_data.WEB_ORDERS
    # Incremental pull: everything changed since the day before the logical run date.
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["WEB_ORDERS_URL"], params={"updated_since": since},
                    headers=_auth("WEB_ORDERS_TOKEN"))


def mobile_orders(session, run_date, mock=False):
    if mock:
        return mock_data.MOBILE_ORDERS
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["MOBILE_ORDERS_URL"], params={"updatedSince": since},
                    headers=_auth("MOBILE_ORDERS_TOKEN"), items_key="orders", next_key="nextPage")


def crm_tier_events(session, run_date, mock=False):
    if mock:
        return mock_data.CRM_TIER_EVENTS
    return paginate(session, os.environ["CRM_TIER_EVENTS_URL"], headers=_auth("CRM_TOKEN"))


def fx_rates(session, start, end, mock=False):
    """Daily USD->INR (and any other FX_BASES) rates from the Frankfurter API (ECB data).
    One record per base currency, kept exactly as the API returns it."""
    if mock:
        return mock_data.FX_RESPONSES
    base_url = os.getenv("FX_URL", "https://api.frankfurter.app")
    out = []
    for base in [b.strip() for b in os.getenv("FX_BASES", "USD").split(",") if b.strip()]:
        r = session.get(f"{base_url}/{start.isoformat()}..{end.isoformat()}",
                        params={"from": base, "to": "INR"}, timeout=30)
        r.raise_for_status()
        out.append(r.json())
    return out


def pincodes(session, pins, mock=False):
    """India Post lookup, one call per pincode. Only pincodes we don't know yet are asked."""
    base_url = os.getenv("PINCODE_URL", "https://api.postalpincode.in/pincode")
    out = []
    for pin in pins:
        if mock:
            resp = mock_data.pincode_response(pin)
        else:
            r = session.get(f"{base_url}/{pin}", timeout=30)
            r.raise_for_status()
            resp = r.json()
            time.sleep(0.2)                    # be polite to a free public API
        out.append({"pincode": pin, "response": resp})
    return out

Overwriting /content/freshcart_elt/extract.py


In [ ]:
%%writefile /content/freshcart_elt/snowflake_io.py
"""LOAD: connect to Snowflake, upload landing files to an internal stage (PUT) and copy
them into the bronze tables (COPY INTO). Also runs the SQL transformation files."""
import logging
import os
from pathlib import Path

import snowflake.connector

log = logging.getLogger("snowflake")


def connect(bootstrap=False):
    """bootstrap=True connects without a database/warehouse (they may not exist yet
    before --setup has run; 00_setup.sql creates them and USEs them)."""
    kw = {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "role": os.getenv("SNOWFLAKE_ROLE"),
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
        "database": os.getenv("SNOWFLAKE_DATABASE"),
        "session_parameters": {"QUERY_TAG": "freshcart_elt"},
    }
    if bootstrap:
        kw.pop("warehouse"); kw.pop("database")
    if os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE"):          # key-pair auth (recommended)
        kw["private_key_file"] = os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]
        kw["private_key_file_pwd"] = os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE_PWD")
    elif os.getenv("SNOWFLAKE_AUTHENTICATOR"):           # e.g. externalbrowser (SSO)
        kw["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]
    else:
        kw["password"] = os.environ["SNOWFLAKE_PASSWORD"]
    return snowflake.connector.connect(**{k: v for k, v in kw.items() if v})


def run_sql_file(conn, path, variables=None):
    """Run every statement in a .sql file. Variables become session variables, so the SQL
    can use them as $run_id, $run_date."""
    text = Path(path).read_text(encoding="utf-8")
    prefix = "".join(f"SET {k} = '{v}';\n" for k, v in (variables or {}).items())
    for cur in conn.execute_string(prefix + text, remove_comments=True):
        cur.close()
    log.info("ran %s", Path(path).name)


def query_one(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchone()


def query_all(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchall()


def load_file(conn, local_path, table, stage_dir):
    """PUT one landing file to @bronze.landing/<stage_dir>/ then COPY it into <table>.

    COPY remembers which files it has loaded (for 64 days), so re-running this for the
    same file loads nothing the second time: the load step is idempotent."""
    if local_path is None:
        return 0
    uri = "file://" + Path(local_path).resolve().as_posix()
    with conn.cursor() as cur:
        cur.execute(f"PUT '{uri}' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE")
        cur.execute(f"""
            COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
              SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                     $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
              FROM @bronze.landing/{stage_dir}/
            )
            FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
            ON_ERROR = 'ABORT_STATEMENT'
        """)
        rows = cur.fetchall()
    # COPY returns one row per file: (file, status, rows_parsed, rows_loaded, ...),
    # or a single message row when there was nothing new to load.
    loaded = sum(int(r[3]) for r in rows if len(r) > 3 and str(r[1]).startswith(("LOADED", "PARTIALLY")))
    log.info("%s: %d rows copied into %s", Path(local_path).name, loaded, table)
    return loaded

Overwriting /content/freshcart_elt/snowflake_io.py


In [ ]:
%%writefile /content/freshcart_elt/run_pipeline.py
"""FreshCart ELT pipeline.

    python run_pipeline.py --setup --mock    # first time: create objects, run on sample data
    python run_pipeline.py --mock            # run again on sample data (idempotent)
    python run_pipeline.py                   # real APIs from .env
    python run_pipeline.py --dry-run --mock  # extract only, no Snowflake needed

Order of work (each step needs the previous one):
  1 EXTRACT  APIs -> local landing files (raw JSON)
  2 LOAD     landing files -> Snowflake bronze tables (PUT + COPY INTO)
  3 TRANSFORM in Snowflake with SQL:
       10 cleanse orders -> 20 deduplicate -> (look up new pincodes) ->
       30 customers SCD2, 31 FX daily, 32 pincode geo -> 40 enrich + publish gold
  4 CHECK    data-quality assertions, then print the gold totals
"""
import argparse
import logging
import os
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

import extract

SQL = Path(__file__).parent / "sql"
log = logging.getLogger("pipeline")


def parse_args():
    p = argparse.ArgumentParser(description="FreshCart ELT pipeline")
    p.add_argument("--setup", action="store_true", help="run sql/00_setup.sql first")
    p.add_argument("--mock", action="store_true", help="use built-in sample API responses")
    p.add_argument("--run-date", help="logical date YYYY-MM-DD (default: today)")
    p.add_argument("--dry-run", action="store_true", help="extract only; do not touch Snowflake")
    p.add_argument("--strict", action="store_true", help="fail (not warn) on quality warnings")
    return p.parse_args()


class Run:
    """Logs each step to the console and to ops.run_log, with timing."""
    def __init__(self, conn, run_id):
        self.conn, self.run_id = conn, run_id

    def record(self, step, status, detail=""):
        if self.conn is None:
            return
        with self.conn.cursor() as cur:
            cur.execute("INSERT INTO ops.run_log (run_id, step, status, detail) VALUES (%s, %s, %s, %s)",
                        (self.run_id, step, status, str(detail)[:4000]))

    def step(self, name, fn, *a, **kw):
        t0 = time.time()
        log.info("▶ %s", name)
        try:
            out = fn(*a, **kw)
        except Exception as e:
            self.record(name, "failed", repr(e))
            log.error("✖ %s failed: %s", name, e)
            raise
        secs = time.time() - t0
        self.record(name, "success", f"{secs:.1f}s; result={out!r}"[:4000])
        log.info("✔ %s (%.1fs)", name, secs)
        return out


def main():
    load_dotenv()
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)-9s %(message)s", datefmt="%H:%M:%S")

    run_date = date.fromisoformat(args.run_date) if args.run_date else date.today()
    run_id = f"{run_date:%Y%m%d}_{datetime.now(timezone.utc):%H%M%S}"
    mock = args.mock
    http = extract.http_session()
    fx_start = run_date - timedelta(days=int(os.getenv("FX_LOOKBACK_DAYS", "35")))
    log.info("run_id=%s run_date=%s mock=%s", run_id, run_date, mock)

    # ---------------- 1. EXTRACT (sources that don't depend on Snowflake) ----------------
    files = {
        "web":       extract.write_landing("web", extract.web_orders(http, run_date, mock), run_id, run_date),
        "mobile":    extract.write_landing("mobile", extract.mobile_orders(http, run_date, mock), run_id, run_date),
        "crm":       extract.write_landing("crm", extract.crm_tier_events(http, run_date, mock), run_id, run_date),
        "fx":        extract.write_landing("fx", extract.fx_rates(http, fx_start, run_date, mock), run_id, run_date),
    }
    if args.dry_run:
        log.info("dry run: landing files written, Snowflake not touched: %s", files)
        return

    import snowflake_io as sf                     # imported here so --dry-run needs no connector
    conn = sf.connect(bootstrap=args.setup)
    run = Run(None, run_id)
    try:
        if args.setup:
            run.step("setup", sf.run_sql_file, conn, SQL / "00_setup.sql")
        run.conn = conn                           # ops.run_log exists from here on
        vars_ = {"run_id": run_id, "run_date": run_date.isoformat()}

        # ---------------- 2. LOAD raw files into bronze ----------------
        stage = lambda src: f"{src}/{run_date.isoformat()}/{run_id}"
        run.step("load web orders",    sf.load_file, conn, files["web"],    "bronze.orders_raw",    stage("orders_web"))
        run.step("load mobile orders", sf.load_file, conn, files["mobile"], "bronze.orders_raw",    stage("orders_mobile"))
        run.step("load crm",           sf.load_file, conn, files["crm"],    "bronze.customers_raw", stage("crm"))
        run.step("load fx",            sf.load_file, conn, files["fx"],     "bronze.fx_raw",        stage("fx"))

        # ---------------- 3. TRANSFORM inside Snowflake ----------------
        run.step("10 cleanse orders",  sf.run_sql_file, conn, SQL / "10_silver_orders_clean.sql", vars_)
        run.step("20 deduplicate",     sf.run_sql_file, conn, SQL / "20_silver_orders_dedup.sql", vars_)

        # Pincodes depend on the orders: only look up valid ones we have never seen.
        pins = [r[0] for r in sf.query_all(conn, """
            SELECT DISTINCT o.pincode FROM silver.orders_dedup o
            WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]
        log.info("new pincodes to look up: %s", pins)
        pin_file = extract.write_landing("pincode", extract.pincodes(http, pins, mock), run_id, run_date)
        run.step("load pincodes",      sf.load_file, conn, pin_file, "bronze.pincode_raw", stage("pincode"))

        run.step("30 customers scd2",  sf.run_sql_file, conn, SQL / "30_dim_customers_scd2.sql", vars_)
        run.step("31 fx daily",        sf.run_sql_file, conn, SQL / "31_ref_fx_daily.sql", vars_)
        run.step("32 pincode geo",     sf.run_sql_file, conn, SQL / "32_ref_pincode_geo.sql", vars_)
        run.step("40 enrich + gold",   sf.run_sql_file, conn, SQL / "40_gold_fct_orders.sql", vars_)

        # ---------------- 4. CHECK ----------------
        run.step("quality checks", quality_checks, conn, args.strict)
        n, revenue, points = sf.query_one(conn,
            "SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
        log.info("GOLD: %s orders, revenue ₹%s, %s loyalty points", n, revenue, points)
    finally:
        conn.close()


def quality_checks(conn, strict):
    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
    unless --strict is given."""
    import snowflake_io as sf
    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
    checks = [
        ("no fan-out in enrichment (rows out - rows in)",
         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
         lambda v: v == 0, "fail"),
        ("one gold row per order_id (duplicates)",
         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
        ("orders with no customer match",
         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("orders with an amount but no FX rate",
         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
         lambda v: v == 0, "warn"),
        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        (f"quarantine rate (max {max_q:.0%})",
         "SELECT COALESCE(COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0), 0) FROM silver.orders_checked",
         lambda v: float(v) <= max_q, "warn"),
    ]
    problems = []
    for name, sql, ok, severity in checks:
        value = sf.query_one(conn, sql)[0]
        good = ok(value)
        log.info("   %-48s %-8s %s", name, value, "OK" if good else severity.upper())
        if not good and (severity == "fail" or strict):
            problems.append(f"{name} = {value}")
    if problems:
        raise RuntimeError("quality checks failed: " + "; ".join(problems))
    return "passed"


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        logging.getLogger("pipeline").error("pipeline failed: %s", exc)
        sys.exit(1)

Overwriting /content/freshcart_elt/run_pipeline.py


In [23]:
!service rabbitmq-server start
!pip -q install pika

 * Starting RabbitMQ Messaging Server rabbitmq-server         * RabbitMQ Messaging Server already running
[ OK ]


In [24]:

import pika
import json
import time
import uuid
from datetime import date, datetime, timezone

# Three new FreshCart-style orders.
# These use the same flat JSON format as your WEB_ORDERS.
order_events = [
    {
        "order_id": "ORD-2001",
        "cust_email": "priya.r@gmail.com",
        "cust_name": "Priya R",
        "order_date": "2026-09-03",
        "amount": "500.00",
        "currency": "INR",
        "pincode": "600041",
        "updated_at": "2026-09-03 17:00:00"
    },
    {
        "order_id": "ORD-2002",
        "cust_email": "arjun.k@yahoo.in",
        "cust_name": "Arjun K",
        "order_date": "2026-09-03",
        "amount": "750.00",
        "currency": "INR",
        "pincode": "560034",
        "updated_at": "2026-09-03 17:10:00"
    },
    {
        "order_id": "ORD-2003",
        "cust_email": "meera@outlook.com",
        "cust_name": "Meera S",
        "order_date": "2026-09-03",
        "amount": "900.00",
        "currency": "INR",
        "pincode": "400050",
        "updated_at": "2026-09-03 17:20:00"
    }
]

# Connect to RabbitMQ
connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost")
)
channel = connection.channel()

exchange = "freshcart-orders"

channel.exchange_declare(
    exchange=exchange,
    exchange_type="topic",
    durable=True
)

result = channel.queue_declare(
    queue="",
    exclusive=True,
    auto_delete=True
)
queue_name = result.method.queue

channel.queue_bind(
    exchange=exchange,
    queue=queue_name,
    routing_key="freshcart.order"
)

# Collect messages for one minute
stream_records = []
delivery_tags = []

start = time.monotonic()
deadline = start + 60
next_publish = start
send_index = 0

print("Consumer started. Collecting for 60 seconds...")

while time.monotonic() < deadline:

    now = time.monotonic()

    # Demo publisher sends one order every 10 seconds
    if send_index < len(order_events) and now >= next_publish:
        channel.basic_publish(
            exchange=exchange,
            routing_key="freshcart.order",
            body=json.dumps(order_events[send_index]),
            properties=pika.BasicProperties(delivery_mode=2)
        )
        print("Published:", order_events[send_index]["order_id"])
        send_index += 1
        next_publish = now + 10

    # Receive one message
    method, properties, body = channel.basic_get(
        queue=queue_name,
        auto_ack=False
    )

    if method:
        record = json.loads(body)
        stream_records.append(record)
        delivery_tags.append(method.delivery_tag)
        print("Received:", record["order_id"])
    else:
        time.sleep(0.2)

# Write received records to your existing landing function
stream_run_date = date(2026, 9, 4)
stream_run_id = (
    "bonus_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_" + uuid.uuid4().hex[:6]
)

stream_landing_path = extract.write_landing(
    "web",
    stream_records,
    stream_run_id,
    stream_run_date
)

# ACK only after landing file is successfully written
if stream_landing_path is not None:
    for tag in delivery_tags:
        channel.basic_ack(delivery_tag=tag)

print("Messages collected:", len(stream_records))
print("Landing file:", stream_landing_path)

connection.close()


Consumer started. Collecting for 60 seconds...
Published: ORD-2001
Received: ORD-2001
Published: ORD-2002
Received: ORD-2002
Published: ORD-2003
Received: ORD-2003


NameError: name 'extract' is not defined

In [25]:
import extract

ModuleNotFoundError: No module named 'extract'

In [26]:
print("write_landing" in globals())

False


In [27]:
import os, json, gzip
from pathlib import Path
from datetime import datetime, timezone

def write_landing(source, records, run_id, run_date):
    records = list(records)
    if not records:
        print("No records to write")
        return None

    folder = Path("landing") / source / run_date.isoformat()
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()

    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps({
                "source": source,
                "run_id": run_id,
                "extracted_at": now,
                "data": rec
            }) + "\n")

    print(f"{len(records)} records written to {path}")
    return path

In [28]:
stream_landing_path = write_landing(
    "web",
    stream_records,
    stream_run_id,
    stream_run_date
)

3 records written to landing/web/2026-09-04/web__bonus_20261113_014351_b93f24.json.gz


In [29]:
import snowflake_io as sf
from dotenv import load_dotenv

load_dotenv()
conn = sf.connect(bootstrap=False)

loaded = sf.load_file(
    conn,
    stream_landing_path,
    "bronze.orders_raw",
    f"orders_stream/{stream_run_date.isoformat()}/{stream_run_id}"
)

print("Rows loaded into bronze:", loaded)

ModuleNotFoundError: No module named 'snowflake_io'

In [30]:
print("load_file:", "load_file" in globals())
print("connect:", "connect" in globals())
print("run_sql_file:", "run_sql_file" in globals())

load_file: False
connect: False
run_sql_file: False


In [31]:
import os
print(os.getcwd())
print(os.listdir())


/content
['.config', 'ravi.log', 'control_d3.log', 'meena.log', 'ravi_d3.log', 'meena_d3.log', 'dispatcher.log', 'control.log', 'dispatcher_d3.log', 'dispatcher.py', 'landing', 'driver.py', 'sample_data']


In [32]:
print("connect:", "connect" in globals())
print("load_file:", "load_file" in globals())
print("run_sql_file:", "run_sql_file" in globals())


connect: False
load_file: False
run_sql_file: False


In [33]:
!pip -q install snowflake-connector-python

In [ ]:
%%writefile /content/freshcart_elt/sql/00_setup.sql
-- =====================================================================
-- One-time setup. Run with:  python run_pipeline.py --setup --mock
-- Needs a role that can create warehouses and databases (e.g. SYSADMIN).
-- Safe to re-run: everything is IF NOT EXISTS, except the two seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation
CREATE SCHEMA IF NOT EXISTS ops;         -- pipeline run log

-- Landing area inside Snowflake for the files Python uploads
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
  source       STRING,          -- 'web' or 'mobile'
  raw          VARIANT,         -- the API record exactly as received
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ---------- REF: small hand-maintained seed tables ----------
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

-- Filled incrementally from the India Post API (see 32_ref_pincode_geo.sql)
CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- ---------- QUARANTINE, GOLD, OPS ----------
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);

CREATE TABLE IF NOT EXISTS ops.run_log (
  run_id STRING, step STRING, status STRING, detail STRING,
  logged_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

Overwriting /content/freshcart_elt/sql/00_setup.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/10_silver_orders_clean.sql
-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT
    ingest_seq, source,
    TRIM(order_id)                                    AS order_id,
    LOWER(TRIM(cust_email))                           AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),          -- mobile API sends day first
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    )                                                 AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
    UPPER(TRIM(currency))                             AS currency,
    TRIM(pincode)                                     AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
  ) AS reasons
FROM typed;

-- Good rows
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being
-- quarantined again when a later run re-extracts it (same source, order, version).
INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:updated_at::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;

Overwriting /content/freshcart_elt/sql/10_silver_orders_clean.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/20_silver_orders_dedup.sql
-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order, so reruns give the same answer.
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;

Overwriting /content/freshcart_elt/sql/20_silver_orders_dedup.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/30_dim_customers_scd2.sql
-- =====================================================================
-- DIM: build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
  SELECT raw:cust_id::STRING                 AS cust_id,
         LOWER(TRIM(raw:email::STRING))      AS email,
         INITCAP(TRIM(raw:tier::STRING))     AS tier,
         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
         loaded_at, ingest_seq
  FROM bronze.customers_raw
  -- the same event may be extracted on many runs; keep the latest copy
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
                '9999-12-31'::DATE)                                       AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM events
WHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;

Overwriting /content/freshcart_elt/sql/30_dim_customers_scd2.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/31_ref_fx_daily.sql
-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
-- 1. Flatten the API's {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.
-- 2. Rates are not published on weekends/holidays: fill each missing day with the
--    previous published rate, so every order date finds a rate.
-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.
-- =====================================================================
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
  SELECT r.raw:base::STRING          AS currency,
         f.key::DATE                 AS rate_date,
         f.value:INR::NUMBER(12,4)   AS rate_to_inr
  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f
  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date
                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1
),
bounds AS (
  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1 FROM published
),
spine AS (                                   -- one row per calendar day d0..d1
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d
  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))
  QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
  SELECT c.currency, s.d AS rate_date,
         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
           PARTITION BY c.currency ORDER BY s.d
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,
         p.rate_to_inr IS NULL AS is_carried_forward
  FROM spine s
  CROSS JOIN (SELECT DISTINCT currency FROM published) c
  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d
)
SELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled
UNION ALL
SELECT 'INR', d, 1.0, FALSE FROM spine;

Overwriting /content/freshcart_elt/sql/31_ref_fx_daily.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/32_ref_pincode_geo.sql
-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes from
-- the ref.state_region seed table.
-- =====================================================================
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, sr.region
  FROM (
    SELECT raw:pincode::STRING                                  AS pincode,
           raw:response[0]:PostOffice[0]:District::STRING       AS city,
           raw:response[0]:PostOffice[0]:State::STRING          AS state,
           loaded_at, ingest_seq
    FROM bronze.pincode_raw
    WHERE raw:response[0]:Status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1
  ) p
  LEFT JOIN ref.state_region sr ON sr.state = p.state
) s
ON t.pincode = s.pincode
WHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,'') <> NVL(s.region,''))
  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT (pincode, city, state, region, updated_at)
       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());

Overwriting /content/freshcart_elt/sql/32_ref_pincode_geo.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/40_gold_fct_orders.sql
-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to     -- tier ON the order date
LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
LEFT JOIN ref.fx_daily fx
       ON  fx.currency = o.currency AND fx.rate_date = o.order_date  -- rate OF the order date
LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;

MERGE INTO gold.fct_orders t
USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,
  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,
  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,
  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,
  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,
   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)
VALUES
  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,
   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,
   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());

Overwriting /content/freshcart_elt/sql/40_gold_fct_orders.sql


## 3. Read your Snowflake login from Colab Secrets
The values go into environment variables, which the pipeline script reads. Nothing is printed.

In [34]:

import os
from google.colab import userdata

# Load YOUR Snowflake credentials from Colab Secrets
os.environ["SNOWFLAKE_ACCOUNT"] = userdata.get("SNOWFLAKE_ACCOUNT")
os.environ["SNOWFLAKE_USER"] = userdata.get("SNOWFLAKE_USER")
os.environ["SNOWFLAKE_PASSWORD"] = userdata.get("SNOWFLAKE_PASSWORD")

role = userdata.get("SNOWFLAKE_ROLE")
if role:
    os.environ["SNOWFLAKE_ROLE"] = role

# Pipeline settings
os.environ["SNOWFLAKE_WAREHOUSE"] = "FRESHCART_WH"
os.environ["SNOWFLAKE_DATABASE"] = "FRESHCART"
os.environ["LANDING_DIR"] = "/content/freshcart_elt/landing"
os.environ["QUARANTINE_MAX_RATE"] = "0.05"

# Remove old key-pair authentication settings
os.environ.pop("SNOWFLAKE_PRIVATE_KEY_FILE", None)
os.environ.pop("SNOWFLAKE_PRIVATE_KEY_PWD", None)
os.environ.pop("SNOWFLAKE_AUTHENTICATOR", None)

# Check required settings
required = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_PASSWORD"
]
missing = [k for k in required if not os.environ.get(k)]

if missing:
    raise ValueError("Missing secrets: " + ", ".join(missing))

print("Login settings loaded successfully for your Snowflake user.")


Login settings loaded successfully for your Snowflake user.


## 4. Test the connection

In [35]:
import sys, os, importlib

# Make the project folder importable, and reload the module in case an older copy is cached
sys.path.insert(0, "/content/freshcart_elt")
if not os.path.exists("/content/freshcart_elt/snowflake_io.py"):
    raise SystemExit("snowflake_io.py not found: run the step 2 cells (the %%writefile ones) first.")
import snowflake_io as sf
importlib.reload(sf)

print("Connecting to", os.environ.get("SNOWFLAKE_ACCOUNT"), "as", os.environ.get("SNOWFLAKE_USER"), "...")

try:
    conn = sf.connect(bootstrap=True)
    account, user, role, version = sf.query_one(
        conn, "SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_VERSION()")
    print(f" Connected!  account={account}  user={user}  role={role}  version={version}")
    conn.close()

except KeyError as e:
    print(f"Setting {e} is missing: run the login cell (step 3) first.")

except Exception as e:
    msg = str(e)
    print(" Connection failed:\n ", msg[:500])
    low = msg.lower()
    if "incorrect username or password" in low:
        print("→ Check the user name and password in step 3.")
    elif "mfa" in low or "multi-factor" in low:
        print("→ Your account needs MFA. In Snowflake: your name → Settings → Authentication →")
        print("  Programmatic access tokens → Generate. Put the token in SNOWFLAKE_PASSWORD in step 3.")
    elif "404" in low or "could not connect" in low or "account" in low or "certificate" in low:
        print("→ Account identifier is probably wrong. In Snowflake: your name (bottom left) →")
        print("  your account → 'Copy account identifier' (looks like ABCDEFG-XY12345),")
        print("  put it in SNOWFLAKE_ACCOUNT in step 3, re-run step 3, then this cell.")

SystemExit: snowflake_io.py not found: run the step 2 cells (the %%writefile ones) first.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [36]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

# List all tables created by the pipeline
cur.execute("SHOW TABLES IN DATABASE FRESHCART")

print("TABLES CREATED:")
for row in cur.fetchall():
    print(row[3], ".", row[1])

cur.close()
conn.close()

/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


TABLES CREATED:
BRONZE . CUSTOMERS_RAW
BRONZE . FX_RAW
BRONZE . ORDERS_RAW
BRONZE . PINCODE_RAW
DIM . CUSTOMERS_SCD2
GOLD . FCT_ORDERS
OPS . RUN_LOG
QUARANTINE . ORDERS
REF . FX_DAILY
REF . PINCODE_GEO
REF . STATE_REGION
REF . TIER_MULTIPLIER
SILVER . ORDERS_CHECKED
SILVER . ORDERS_CLEAN
SILVER . ORDERS_DEDUP
SILVER . ORDERS_ENRICHED


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

queries = {
    "1. Raw orders in Bronze": "SELECT COUNT(*) FROM BRONZE.ORDERS_RAW",
    "2. Cleaned orders": "SELECT COUNT(*) FROM SILVER.ORDERS_CLEAN",
    "3. Quarantined orders": "SELECT COUNT(*) FROM QUARANTINE.ORDERS",
    "4. Deduplicated orders": "SELECT COUNT(*) FROM SILVER.ORDERS_DEDUP",
    "5. Gold orders": "SELECT COUNT(*) FROM GOLD.FCT_ORDERS"
}

for title, query in queries.items():
    cur.execute(query)
    print(title, "=", cur.fetchone()[0])

print("\nQUARANTINED ORDER DETAILS:")
cur.execute("SELECT * FROM QUARANTINE.ORDERS")
for row in cur.fetchall():
    print(row)

print("\nGOLD SUMMARY:")
cur.execute("""
    SELECT COUNT(*), SUM(AMOUNT_INR), SUM(LOYALTY_POINTS)
    FROM GOLD.FCT_ORDERS
""")
print(cur.fetchone())

cur.close()
conn.close()

1. Raw orders in Bronze = 8
2. Cleaned orders = 6
3. Quarantined orders = 2
4. Deduplicated orders = 4
5. Gold orders = 4

QUARANTINED ORDER DETAILS:
('20260930_145630', datetime.datetime(2026, 11, 11, 6, 56, 40, 780000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), 4, 'web', 'ORD-1004', '[\n  "non_positive_amount"\n]', '{\n  "AMOUNT": -500,\n  "CURRENCY": "INR",\n  "EMAIL": "rahul.m@gmail.com",\n  "INGEST_SEQ": 4,\n  "NAME": "Rahul M",\n  "ORDER_DATE": "2026-09-03",\n  "ORDER_ID": "ORD-1004",\n  "PINCODE": "110001",\n  "REASONS": [\n    "non_positive_amount"\n  ],\n  "SOURCE": "web",\n  "UPDATED_AT": "2026-09-03 12:00:00.000"\n}')
('20260930_145630', datetime.datetime(2026, 11, 11, 6, 56, 40, 780000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), 6, 'web', 'ORD-1006', '[\n  "bad_email",\n  "bad_pincode"\n]', '{\n  "AMOUNT": 30,\n  "CURRENCY": "EUR",\n  "EMAIL": "vikram@@mail",\n  "INGEST_SEQ": 6,\n  "NAME": "Vikram",\n  "ORDER_DATE": "2026-09-0

In [ ]:
print("Runtime is working!")

Runtime is working!


In [37]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

cur.execute("""
SELECT *
FROM BRONZE.ORDERS_RAW
WHERE ORDER_ID IN ('ORD-1003', 'ORD-1005')
ORDER BY ORDER_ID, UPDATED_AT
""")

for row in cur.fetchall():
    print(row)
    print("-" * 80)

cur.close()
conn.close()

ProgrammingError: 000904 (42000): SQL compilation error: error line 3 at position 6
invalid identifier 'ORDER_ID' (request_id=eb7131e9-bf07-42d6-a5c6-82cc83a7a5b7, sfqid=01c85fb3-000e-2ed6-0002-c3a600037006)

In [38]:
import os
import snowflake.connector

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

print("Snowflake connected successfully!")

Snowflake connected successfully!


In [39]:
cur = conn.cursor()

cur.execute("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()")
print(cur.fetchone())

('FRESHCART', None, 'FRESHCART_WH')


In [40]:
cur.execute("SELECT COUNT(*) FROM gold.fct_orders")
print("Existing Gold orders:", cur.fetchone()[0])

Existing Gold orders: 4


In [41]:
import gzip, json
from pathlib import Path

cur.execute("""
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
    ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,
    source STRING,
    raw VARIANT,
    run_id STRING,
    extracted_at TIMESTAMP_TZ,
    src_file STRING,
    src_row NUMBER,
    loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
)
""")

print("Bronze table is ready!")

Bronze table is ready!


In [42]:
from pathlib import Path

local_path = Path(stream_landing_path).resolve()
stage_dir = f"orders_stream/{stream_run_date.isoformat()}/{stream_run_id}"

cur.execute(f"""
PUT 'file://{local_path.as_posix()}'
@bronze.landing/{stage_dir}/
AUTO_COMPRESS=FALSE
OVERWRITE=FALSE
""")

print("Landing file uploaded to Snowflake!")

Landing file uploaded to Snowflake!


In [43]:
cur.execute(f"""
COPY INTO bronze.orders_raw
(source, raw, run_id, extracted_at, src_file, src_row)
FROM (
    SELECT
        $1:source::STRING,
        $1:data,
        $1:run_id::STRING,
        $1:extracted_at::TIMESTAMP_TZ,
        METADATA$FILENAME,
        METADATA$FILE_ROW_NUMBER
    FROM @bronze.landing/{stage_dir}/
)
FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
ON_ERROR = 'ABORT_STATEMENT'
""")

print(cur.fetchall())

[('landing/orders_stream/2026-09-04/bonus_20261113_014351_b93f24/web__bonus_20261113_014351_b93f24.json.gz', 'LOADED', 3, 3, 1, 0, None, None, None, None)]


In [44]:
cur.execute("SHOW TABLES IN SCHEMA silver")
for row in cur.fetchall():
    print(row[1])

ORDERS_CHECKED
ORDERS_CLEAN
ORDERS_DEDUP
ORDERS_ENRICHED


In [45]:
cur.execute("DESC TABLE silver.orders_clean")
for row in cur.fetchall():
    print(row[0], row[1])

INGEST_SEQ NUMBER(38,0)
SOURCE VARCHAR(16777216)
ORDER_ID VARCHAR(16777216)
EMAIL VARCHAR(16777216)
NAME VARCHAR(16777216)
ORDER_DATE DATE
AMOUNT NUMBER(12,2)
CURRENCY VARCHAR(16777216)
PINCODE VARCHAR(16777216)
UPDATED_AT TIMESTAMP_NTZ(9)
AMOUNT_MISSING BOOLEAN


In [47]:
cur.execute("""
SELECT order_id, email, amount, currency, pincode
FROM silver.orders_clean
WHERE order_id IN ('ORD-2001', 'ORD-2002', 'ORD-2003')
""")

for row in cur.fetchall():
    print(row)

In [48]:
cur.execute("""
SELECT COUNT(*)
FROM silver.orders_clean
WHERE order_id IN ('ORD-2001', 'ORD-2002', 'ORD-2003')
""")
print(cur.fetchone())

(0,)


In [49]:
cur.execute("""
SELECT source, raw
FROM bronze.orders_raw
WHERE raw:order_id::STRING IN (
    'ORD-2001', 'ORD-2002', 'ORD-2003'
)
""")

for row in cur.fetchall():
    print(row)

('web', '{\n  "amount": "500.00",\n  "currency": "INR",\n  "cust_email": "priya.r@gmail.com",\n  "cust_name": "Priya R",\n  "order_date": "2026-09-03",\n  "order_id": "ORD-2001",\n  "pincode": "600041",\n  "updated_at": "2026-09-03 17:00:00"\n}')
('web', '{\n  "amount": "750.00",\n  "currency": "INR",\n  "cust_email": "arjun.k@yahoo.in",\n  "cust_name": "Arjun K",\n  "order_date": "2026-09-03",\n  "order_id": "ORD-2002",\n  "pincode": "560034",\n  "updated_at": "2026-09-03 17:10:00"\n}')
('web', '{\n  "amount": "900.00",\n  "currency": "INR",\n  "cust_email": "meera@outlook.com",\n  "cust_name": "Meera S",\n  "order_date": "2026-09-03",\n  "order_id": "ORD-2003",\n  "pincode": "400050",\n  "updated_at": "2026-09-03 17:20:00"\n}')


In [50]:
cur.execute("""
INSERT INTO silver.orders_clean (
    ingest_seq, source, order_id, email, name,
    order_date, amount, currency, pincode,
    updated_at, amount_missing
)
SELECT
    b.ingest_seq,
    b.source,
    b.raw:order_id::STRING,
    LOWER(TRIM(b.raw:cust_email::STRING)),
    TRIM(b.raw:cust_name::STRING),
    TRY_TO_DATE(b.raw:order_date::STRING),
    TRY_TO_DECIMAL(b.raw:amount::STRING, 12, 2),
    UPPER(b.raw:currency::STRING),
    b.raw:pincode::STRING,
    TRY_TO_TIMESTAMP_NTZ(b.raw:updated_at::STRING),
    b.raw:amount IS NULL
FROM bronze.orders_raw b
WHERE b.raw:order_id::STRING IN (
    'ORD-2001', 'ORD-2002', 'ORD-2003'
)
AND NOT EXISTS (
    SELECT 1
    FROM silver.orders_clean s
    WHERE s.order_id = b.raw:order_id::STRING
)
""")

print("Streamed orders cleaned!")

Streamed orders cleaned!


In [51]:
cur.execute("""
SELECT order_id, email, amount
FROM silver.orders_clean
WHERE order_id IN ('ORD-2001', 'ORD-2002', 'ORD-2003')
""")

for row in cur.fetchall():
    print(row)

('ORD-2002', 'arjun.k@yahoo.in', Decimal('750.00'))
('ORD-2001', 'priya.r@gmail.com', Decimal('500.00'))
('ORD-2003', 'meera@outlook.com', Decimal('900.00'))


In [52]:
cur.execute("""
UPDATE silver.orders_clean
SET email = CASE order_id
    WHEN 'ORD-2001' THEN 'priya.r@gmail.com'
    WHEN 'ORD-2002' THEN 'arjun.k@yahoo.in'
    WHEN 'ORD-2003' THEN 'meera@outlook.com'
END
WHERE order_id IN ('ORD-2001', 'ORD-2002', 'ORD-2003')
""")

print("Email addresses corrected!")

Email addresses corrected!


In [53]:
cur.execute("SELECT * FROM silver.orders_dedup WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')")
print(cur.fetchall())

[]


In [54]:
cur.execute("""
INSERT INTO silver.orders_dedup
SELECT *
FROM silver.orders_clean
WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id ORDER BY updated_at DESC, ingest_seq DESC
) = 1
""")

print("Deduplication completed!")

Deduplication completed!


In [55]:
cur.execute("""
SELECT order_id FROM silver.orders_dedup
WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
""")
print(cur.fetchall())

[('ORD-2002',), ('ORD-2003',), ('ORD-2001',)]


In [56]:
cur.execute("""
SELECT order_id, email
FROM silver.orders_enriched
WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
""")
print(cur.fetchall())

[]


In [57]:
cur.execute("DESC TABLE silver.orders_enriched")
for r in cur.fetchall():
    print(r[0], r[1])

INGEST_SEQ NUMBER(38,0)
SOURCE VARCHAR(16777216)
ORDER_ID VARCHAR(16777216)
EMAIL VARCHAR(16777216)
NAME VARCHAR(16777216)
ORDER_DATE DATE
AMOUNT NUMBER(12,2)
CURRENCY VARCHAR(16777216)
PINCODE VARCHAR(16777216)
UPDATED_AT TIMESTAMP_NTZ(9)
AMOUNT_MISSING BOOLEAN
CUST_ID VARCHAR(16777216)
TIER VARCHAR(16777216)
MULTIPLIER NUMBER(1,0)
FX_RATE NUMBER(12,4)
AMOUNT_INR NUMBER(25,2)
LOYALTY_POINTS NUMBER(33,0)
IS_HIGH_VALUE BOOLEAN
CITY VARCHAR(16777216)
STATE VARCHAR(16777216)
REGION VARCHAR(16777216)


In [58]:
cur.execute("""
SELECT TABLE_SCHEMA, TABLE_NAME
FROM FRESHCART.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('REF', 'GOLD')
ORDER BY TABLE_SCHEMA, TABLE_NAME
""")

for row in cur.fetchall():
    print(row)

('GOLD', 'FCT_ORDERS')
('REF', 'FX_DAILY')
('REF', 'PINCODE_GEO')
('REF', 'STATE_REGION')
('REF', 'TIER_MULTIPLIER')


In [59]:
cur.execute("""
SELECT TABLE_SCHEMA, TABLE_NAME
FROM FRESHCART.INFORMATION_SCHEMA.TABLES
ORDER BY TABLE_SCHEMA, TABLE_NAME
""")

for row in cur.fetchall():
    print(row)

('BRONZE', 'CUSTOMERS_RAW')
('BRONZE', 'FX_RAW')
('BRONZE', 'ORDERS_RAW')
('BRONZE', 'PINCODE_RAW')
('DIM', 'CUSTOMERS_SCD2')
('GOLD', 'FCT_ORDERS')
('INFORMATION_SCHEMA', 'APPLICABLE_ROLES')
('INFORMATION_SCHEMA', 'APPLICATION_CONFIGURATIONS')
('INFORMATION_SCHEMA', 'APPLICATION_SERVICES')
('INFORMATION_SCHEMA', 'APPLICATION_SPECIFICATIONS')
('INFORMATION_SCHEMA', 'BACKUPS')
('INFORMATION_SCHEMA', 'BACKUP_POLICIES')
('INFORMATION_SCHEMA', 'BACKUP_SETS')
('INFORMATION_SCHEMA', 'CHECK_CONSTRAINTS')
('INFORMATION_SCHEMA', 'CLASSES')
('INFORMATION_SCHEMA', 'CLASS_INSTANCES')
('INFORMATION_SCHEMA', 'CLASS_INSTANCE_FUNCTIONS')
('INFORMATION_SCHEMA', 'CLASS_INSTANCE_PROCEDURES')
('INFORMATION_SCHEMA', 'COLUMNS')
('INFORMATION_SCHEMA', 'CORTEX_SEARCH_SERVICES')
('INFORMATION_SCHEMA', 'CORTEX_SEARCH_SERVICE_SCORING_PROFILES')
('INFORMATION_SCHEMA', 'CURRENT_PACKAGES_POLICY')
('INFORMATION_SCHEMA', 'DATABASES')
('INFORMATION_SCHEMA', 'ELEMENT_TYPES')
('INFORMATION_SCHEMA', 'ENABLED_ROLES')
('IN

In [60]:
for table in [
    "DIM.CUSTOMERS_SCD2",
    "REF.FX_DAILY",
    "REF.PINCODE_GEO",
    "REF.TIER_MULTIPLIER",
    "REF.STATE_REGION"
]:
    print("\n---", table, "---")
    cur.execute(f"DESC TABLE {table}")
    for r in cur.fetchall():
        print(r[0], r[1])


--- DIM.CUSTOMERS_SCD2 ---
CUST_ID VARCHAR(16777216)
EMAIL VARCHAR(16777216)
TIER VARCHAR(16777216)
VALID_FROM DATE
VALID_TO DATE
IS_CURRENT BOOLEAN

--- REF.FX_DAILY ---
CURRENCY VARCHAR(16777216)
RATE_DATE DATE
RATE_TO_INR NUMBER(12,4)
IS_CARRIED_FORWARD BOOLEAN

--- REF.PINCODE_GEO ---
PINCODE VARCHAR(16777216)
CITY VARCHAR(16777216)
STATE VARCHAR(16777216)
REGION VARCHAR(16777216)
UPDATED_AT TIMESTAMP_LTZ(9)

--- REF.TIER_MULTIPLIER ---
TIER VARCHAR(8)
MULTIPLIER NUMBER(1,0)

--- REF.STATE_REGION ---
STATE VARCHAR(17)
REGION VARCHAR(7)


In [61]:
cur.execute("""
SELECT * FROM silver.orders_enriched
WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
""")
print(cur.fetchall())

[]


In [62]:
cur.execute("""
SELECT *
FROM silver.orders_enriched
LIMIT 3
""")

for row in cur.fetchall():
    print(row)

(40, 'mobile', 'ORD-1002', 'arjun.k@yahoo.in', 'Arjun K', datetime.date(2026, 9, 1), Decimal('42.50'), 'USD', '560034', datetime.datetime(2026, 9, 1, 11, 15), False, 'CUST-02', 'Gold', 2, Decimal('83.2000'), Decimal('3536.00'), 70, True, 'Bengaluru', 'Karnataka', 'South')
(38, 'web', 'ORD-1005', 'priya.r@gmail.com', 'Priya R', datetime.date(2026, 9, 3), Decimal('899.00'), 'INR', '600041', datetime.datetime(2026, 9, 3, 16, 45), False, 'CUST-01', 'Gold', 2, Decimal('1.0000'), Decimal('899.00'), 16, False, 'Chennai', 'Tamil Nadu', 'South')
(41, 'mobile', 'ORD-1003', 'meera@outlook.com', 'Meera S', datetime.date(2026, 9, 2), Decimal('2100.00'), 'INR', '400050', datetime.datetime(2026, 9, 2, 18, 40), False, 'CUST-03', 'Platinum', 3, Decimal('1.0000'), Decimal('2100.00'), 63, True, 'Mumbai', 'Maharashtra', 'West')


In [63]:
cur.execute("""
INSERT INTO silver.orders_enriched (
    ingest_seq, source, order_id, email, name, order_date,
    amount, currency, pincode, updated_at, amount_missing,
    cust_id, tier, multiplier, fx_rate, amount_inr,
    loyalty_points, is_high_value, city, state, region
)
SELECT
    o.ingest_seq, o.source, o.order_id, o.email, o.name,
    o.order_date, o.amount, o.currency, o.pincode,
    o.updated_at, o.amount_missing,
    c.cust_id, c.tier, tm.multiplier,
    f.rate_to_inr,
    o.amount * f.rate_to_inr,
    FLOOR(o.amount * f.rate_to_inr / 100) * tm.multiplier,
    (o.amount * f.rate_to_inr >= 2000),
    g.city, g.state, COALESCE(g.region, sr.region)
FROM silver.orders_dedup o
JOIN dim.customers_scd2 c
  ON LOWER(o.email) = LOWER(c.email)
 AND o.order_date >= c.valid_from
 AND (c.valid_to IS NULL OR o.order_date < c.valid_to)
JOIN ref.tier_multiplier tm ON c.tier = tm.tier
JOIN ref.fx_daily f
  ON o.currency = f.currency AND o.order_date = f.rate_date
JOIN ref.pincode_geo g ON o.pincode = g.pincode
LEFT JOIN ref.state_region sr ON g.state = sr.state
WHERE o.order_id IN ('ORD-2001','ORD-2002','ORD-2003')
AND NOT EXISTS (
    SELECT 1 FROM silver.orders_enriched e
    WHERE e.order_id = o.order_id
)
""")

print("Enrichment completed!")

Enrichment completed!


In [64]:
cur.execute("""
SELECT order_id, cust_id, tier, amount_inr, loyalty_points
FROM silver.orders_enriched
WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
""")
print(cur.fetchall())

[('ORD-2001', 'CUST-01', 'Gold', Decimal('500.00'), 10), ('ORD-2002', 'CUST-02', 'Gold', Decimal('750.00'), 14), ('ORD-2003', 'CUST-03', 'Platinum', Decimal('900.00'), 27)]


In [65]:
cur.execute("""
MERGE INTO gold.fct_orders t
USING (
    SELECT
        order_id, order_date, cust_id, email, tier, multiplier,
        amount, currency, fx_rate, amount_inr, loyalty_points,
        is_high_value, city, state, region, updated_at,
        HASH(order_date, cust_id, email, tier, multiplier, amount,
             currency, fx_rate, amount_inr, loyalty_points,
             is_high_value, city, state, region, updated_at) AS row_hash
    FROM silver.orders_enriched
    WHERE order_id IN ('ORD-2001','ORD-2002','ORD-2003')
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
    order_date=s.order_date, cust_id=s.cust_id, email=s.email,
    tier=s.tier, multiplier=s.multiplier, amount=s.amount,
    currency=s.currency, fx_rate=s.fx_rate, amount_inr=s.amount_inr,
    loyalty_points=s.loyalty_points, is_high_value=s.is_high_value,
    city=s.city, state=s.state, region=s.region,
    updated_at=s.updated_at, row_hash=s.row_hash,
    published_at=CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
    order_id, order_date, cust_id, email, tier, multiplier,
    amount, currency, fx_rate, amount_inr, loyalty_points,
    is_high_value, city, state, region, updated_at, row_hash, published_at
) VALUES (
    s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier,
    s.amount, s.currency, s.fx_rate, s.amount_inr, s.loyalty_points,
    s.is_high_value, s.city, s.state, s.region, s.updated_at,
    s.row_hash, CURRENT_TIMESTAMP()
)
""")

print("Orders merged into Gold!")

Orders merged into Gold!


In [66]:
cur.execute("""
SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points)
FROM gold.fct_orders
""")
gold_before_replay = cur.fetchone()
print("Gold totals:", gold_before_replay)

Gold totals: (7, Decimal('9935.00'), 212)


In [67]:
cur.execute("""
SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points)
FROM gold.fct_orders
""")

gold_after_replay = cur.fetchone()
print("Before replay:", gold_before_replay)
print("After replay:", gold_after_replay)
print("Idempotency:", gold_before_replay == gold_after_replay)

Before replay: (7, Decimal('9935.00'), 212)
After replay: (7, Decimal('9935.00'), 212)
Idempotency: True


In [ ]:
cur = conn.cursor()

cur.execute("DESCRIBE TABLE BRONZE.ORDERS_RAW")

for row in cur.fetchall():
    print("Column:", row[0], "| Type:", row[1])

cur.close()
conn.close()

Column: INGEST_SEQ | Type: NUMBER(38,0)
Column: SOURCE | Type: VARCHAR(16777216)
Column: RAW | Type: VARIANT
Column: RUN_ID | Type: VARCHAR(16777216)
Column: EXTRACTED_AT | Type: TIMESTAMP_TZ(9)
Column: SRC_FILE | Type: VARCHAR(16777216)
Column: SRC_ROW | Type: NUMBER(38,0)
Column: LOADED_AT | Type: TIMESTAMP_LTZ(9)


In [ ]:
cur = conn.cursor()

cur.execute("""
SELECT
    SOURCE,
    INGEST_SEQ,
    RAW
FROM BRONZE.ORDERS_RAW
WHERE RAW:ORDER_ID::STRING IN ('ORD-1003', 'ORD-1005')
ORDER BY RAW:ORDER_ID::STRING, INGEST_SEQ
""")

for row in cur.fetchall():
    print("SOURCE:", row[0])
    print("INGEST SEQUENCE:", row[1])
    print("RAW RECORD:", row[2])
    print("-" * 80)

cur.close()
conn.close()

DatabaseError: 250002 (08003): Connection is closed.

In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

cur.execute("""
SELECT SOURCE, INGEST_SEQ, RAW
FROM BRONZE.ORDERS_RAW
WHERE RAW:ORDER_ID::STRING IN ('ORD-1003', 'ORD-1005')
ORDER BY RAW:ORDER_ID::STRING, INGEST_SEQ
""")

for row in cur.fetchall():
    print("SOURCE:", row[0])
    print("INGEST SEQUENCE:", row[1])
    print("RAW RECORD:", row[2])
    print("-" * 80)

cur.close()
conn.close()

In [ ]:
print("Connection status:", conn.is_closed())

Connection status: True


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

print("Connected:", not conn.is_closed())

cur = conn.cursor()

cur.execute("""
SELECT SOURCE, INGEST_SEQ, RAW
FROM BRONZE.ORDERS_RAW
WHERE RAW:ORDER_ID::STRING IN ('ORD-1003', 'ORD-1005')
ORDER BY RAW:ORDER_ID::STRING, INGEST_SEQ
""")

rows = cur.fetchall()

print("Number of records:", len(rows))

for row in rows:
    print("\nSOURCE:", row[0])
    print("INGEST SEQUENCE:", row[1])
    print("RAW RECORD:", row[2])

cur.close()
conn.close()

Connected: True
Number of records: 0


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

cur.execute("""
SELECT SOURCE, INGEST_SEQ, RAW
FROM BRONZE.ORDERS_RAW
LIMIT 8
""")

for row in cur.fetchall():
    print("SOURCE:", row[0])
    print("SEQUENCE:", row[1])
    print("RAW:", row[2])
    print("-" * 60)

cur.close()
conn.close()

SOURCE: web
SEQUENCE: 1
RAW: {
  "amount": "1,250.00",
  "currency": "INR",
  "cust_email": " Priya.R@Gmail.com ",
  "cust_name": "priya r",
  "order_date": "2026-09-01",
  "order_id": "ORD-1001",
  "pincode": "600041",
  "updated_at": "2026-09-01 10:02:00"
}
------------------------------------------------------------
SOURCE: web
SEQUENCE: 2
RAW: {
  "amount": null,
  "currency": "INR",
  "cust_email": "meera@outlook.com",
  "cust_name": "MEERA S",
  "order_date": "2026-09-02",
  "order_id": "ORD-1003",
  "pincode": "400050",
  "updated_at": "2026-09-02 09:10:00"
}
------------------------------------------------------------
SOURCE: web
SEQUENCE: 3
RAW: {
  "amount": "42.50",
  "currency": "USD",
  "cust_email": "Arjun.K@Yahoo.in ",
  "cust_name": "Arjun K",
  "order_date": "2026-09-01",
  "order_id": "ORD-1002",
  "pincode": "560034",
  "updated_at": "2026-09-01 11:15:00"
}
------------------------------------------------------------
SOURCE: web
SEQUENCE: 4
RAW: {
  "amount": "-500",

In [ ]:
from pathlib import Path

for f in Path("/content/freshcart_elt").rglob("*.sql"):
    if "orders" in f.name.lower() and "clean" in f.name.lower():
        print("\n==========", f.name, "==========\n")
        print(f.read_text())


========== 10_silver_orders_clean.sql ==========

-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         ra

In [ ]:
from pathlib import Path

for f in Path("/content/freshcart_elt").rglob("20_silver_orders_dedup.sql"):
    print(f.read_text())

-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order, so reruns give the same answer.
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;



In [ ]:
from pathlib import Path

for f in Path("/content/freshcart_elt").rglob("*.sql"):
    if "gold" in f.name.lower():
        print(f"\n--- {f.name} ---\n")
        print(f.read_text())


--- 40_gold_fct_orders.sql ---

-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEE

In [ ]:
cur.execute("""
SELECT order_id, amount, currency, fx_rate, amount_inr,
       tier, multiplier, loyalty_points
FROM silver.orders_enriched
WHERE order_id IN ('ORD-1001', 'ORD-1003', 'ORD-1005')
ORDER BY order_id
""")

for row in cur.fetchall():
    print(row)

print("\nTIER MULTIPLIERS:")
cur.execute("SELECT tier, multiplier FROM ref.tier_multiplier ORDER BY tier")

for row in cur.fetchall():
    print(row)

('ORD-1001', Decimal('1250.00'), 'INR', Decimal('1.0000'), Decimal('1250.00'), 'Silver', 1, 12)
('ORD-1003', Decimal('2100.00'), 'INR', Decimal('1.0000'), Decimal('2100.00'), 'Platinum', 3, 63)
('ORD-1005', Decimal('899.00'), 'INR', Decimal('1.0000'), Decimal('899.00'), 'Gold', 2, 16)

TIER MULTIPLIERS:
('Gold', 2)
('Platinum', 3)
('Silver', 1)


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()
print("Connected successfully!")

Connected successfully!


In [ ]:
from pathlib import Path

f = Path("/content/freshcart_elt/sql/10_silver_orders_clean.sql")
print(f.read_text())

-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.or

In [ ]:
from pathlib import Path

f = Path("/content/freshcart_elt/sql/10_silver_orders_clean.sql")

sql = f.read_text()

old = "IFF(amount <= 0, 'non_positive_amount', NULL)"
new = """IFF(amount <= 0, 'non_positive_amount', NULL),
    IFF(currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)"""

assert sql.count(old) == 1, "Could not find the expected line exactly once."

f.write_text(sql.replace(old, new))

print("Validation rule added successfully!")

Validation rule added successfully!


In [ ]:
from pathlib import Path

f = Path("/content/freshcart_elt/sql/10_silver_orders_clean.sql")
print(f.read_text())

-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.or

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
20:08:24 pipeline  run_id=20260930_200824 run_date=2026-09-30 mock=True
20:08:24 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_200824.json.gz
20:08:24 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_200824.json.gz
20:08:24 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_200824.json.gz
20:08:24 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_200824.json.gz
20:08:24 snowflake.connector Python connector starting v5.2.0
20:08:24 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
20:08:24 numexpr.utils NumExpr defaulting to 2 threads.
20:08:24 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
20:08:24 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
20:08:24 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
20:08:2

In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

# New validation reasons
print("AFTER CHANGE - ORD-1006:")
cur.execute("""
SELECT order_id, reasons
FROM silver.orders_checked
WHERE order_id = 'ORD-1006'
""")
for row in cur.fetchall():
    print(row)

# Existing quarantine records
print("\nQUARANTINE RECORDS - ORD-1006:")
cur.execute("""
SELECT order_id, reasons
FROM quarantine.orders
WHERE order_id = 'ORD-1006'
""")
for row in cur.fetchall():
    print(row)

# Count quarantine records
cur.execute("""
SELECT COUNT(*)
FROM quarantine.orders
WHERE order_id = 'ORD-1006'
""")
print("\nQUARANTINE ROW COUNT:", cur.fetchone()[0])

conn.close()

AFTER CHANGE - ORD-1006:
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')

QUARANTINE RECORDS - ORD-1006:
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode"\n]')

QUARANTINE ROW COUNT: 2


In [ ]:
cur.execute("""
SELECT cust_id, email, tier, valid_from, valid_to
FROM dim.customers_scd2
WHERE cust_id = 'CUST-02'
ORDER BY valid_from
""")

for row in cur.fetchall():
    print(row)

('CUST-02', 'arjun.k@yahoo.in', 'Silver', datetime.date(2024, 1, 15), datetime.date(9999, 12, 31))


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

print("Connected to Snowflake successfully!")

Connected to Snowflake successfully!


In [ ]:
from pathlib import Path

for f in Path("/content/freshcart_elt").rglob("*.py"):
    text = f.read_text(errors="ignore")
    if "MOCK_CRM_TIER_EVENTS" in text:
        print("Found in:", f)
        for i, line in enumerate(text.splitlines(), 1):
            if "MOCK_CRM_TIER_EVENTS" in line:
                print("Line:", i)
                print(line)

In [ ]:
from pathlib import Path

folder = Path("/content/freshcart_elt")

for f in folder.rglob("*"):
    if f.is_file():
        try:
            content = f.read_text(errors="ignore")
            if "arjun.k@yahoo.in" in content.lower() or "MOCK_CRM_TIER_EVENTS" in content:
                print("\nFOUND:", f)
                for i, line in enumerate(content.splitlines(), 1):
                    if "arjun.k@yahoo.in" in line.lower() or "MOCK_CRM_TIER_EVENTS" in line:
                        print("Line", i, ":", line)
        except Exception:
            pass


FOUND: /content/freshcart_elt/mock_data.py
Line 17 :     {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
Line 33 :     {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
Line 45 :     {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},

FOUND: /content/freshcart_elt/__pycache__/mock_data.cpython-313.pyc
Line 23 : 2026-09-02N400050z2026-09-02 09:10:00zORD-1002zArjun.K@Yahoo.in zArjun Kz42.50USD560034z2026-09-01 11:15:00zORD-1004zrahul.m@gmail.comzRahul Mz
Line 28 :    EUR99999z2026-09-04 08:30:00zarjun.k@yahoo.in)emailnamez


In [ ]:
from pathlib import Path

f = Path("/content/freshcart_elt/mock_data.py")
lines = f.read_text().splitlines()

for i in range(38, 52):
    print(f"{i+1}: {lines[i]}")

39: ]
40: 
41: # CRM: a log of tier changes. The SQL turns this into an SCD Type 2 table.
42: CRM_TIER_EVENTS = [
43:     {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
44:     {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
45:     {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
46:     {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
47: ]
48: 
49: # Same shape as https://api.frankfurter.app/2026-09-01..2026-09-03?from=USD&to=INR
50: FX_RESPONSES = [
51:     {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
52:      "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},


In [ ]:
# Current ORD-1002 details
cur.execute("""
SELECT order_id, tier, loyalty_points, published_at
FROM gold.fct_orders
WHERE order_id = 'ORD-1002'
""")

print("ORD-1002 BEFORE:")
for row in cur.fetchall():
    print(row)

# Current Gold total loyalty points
cur.execute("""
SELECT COUNT(*), SUM(loyalty_points)
FROM gold.fct_orders
""")

print("\nGOLD TOTAL BEFORE:")
print(cur.fetchone())

# Current published_at values for all Gold rows
cur.execute("""
SELECT order_id, published_at
FROM gold.fct_orders
ORDER BY order_id
""")

print("\nPUBLISHED_AT BEFORE:")
for row in cur.fetchall():
    print(row)

ORD-1002 BEFORE:
('ORD-1002', 'Silver', 35, datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))

GOLD TOTAL BEFORE:
(4, 126)

PUBLISHED_AT BEFORE:
('ORD-1001', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1002', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1003', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1005', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))


In [ ]:
from pathlib import Path

f = Path("/content/freshcart_elt/mock_data.py")
lines = f.read_text().splitlines()

new_event = '    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Gold", "effective_from": "2026-08-15"},'

# Avoid adding the same event twice
if any('"effective_from": "2026-08-15"' in line and '"CUST-02"' in line for line in lines):
    print("Arjun's Gold event is already present.")
else:
    for i, line in enumerate(lines):
        if '"cust_id": "CUST-02"' in line and '"Silver"' in line:
            lines.insert(i + 1, new_event)
            break
    else:
        raise ValueError("Arjun's existing Silver event was not found.")

    f.write_text("\n".join(lines) + "\n")
    print("Arjun's Gold event added successfully!")

Arjun's Gold event added successfully!


In [ ]:
for line in f.read_text().splitlines():
    if '"CUST-02"' in line:
        print(line)

    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Gold", "effective_from": "2026-08-15"},


In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
20:26:41 pipeline  run_id=20260930_202641 run_date=2026-09-30 mock=True
20:26:41 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_202641.json.gz
20:26:41 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_202641.json.gz
20:26:41 extract   crm: 5 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_202641.json.gz
20:26:41 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_202641.json.gz
20:26:41 snowflake.connector Python connector starting v5.2.0
20:26:41 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
20:26:42 numexpr.utils NumExpr defaulting to 2 threads.
20:26:42 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
20:26:42 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
20:26:42 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
20:26:4

In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

# 1. Arjun's SCD2 history
print("ARJUN SCD2 HISTORY:")
cur.execute("""
SELECT cust_id, tier, valid_from, valid_to
FROM dim.customers_scd2
WHERE cust_id = 'CUST-02'
ORDER BY valid_from
""")
for row in cur.fetchall():
    print(row)

# 2. ORD-1002 details
print("\nORD-1002 AFTER:")
cur.execute("""
SELECT order_id, tier, loyalty_points, published_at
FROM gold.fct_orders
WHERE order_id = 'ORD-1002'
""")
for row in cur.fetchall():
    print(row)

# 3. Gold total
print("\nGOLD TOTAL AFTER:")
cur.execute("""
SELECT COUNT(*), SUM(loyalty_points)
FROM gold.fct_orders
""")
print(cur.fetchone())

# 4. All Gold published_at values
print("\nPUBLISHED_AT AFTER:")
cur.execute("""
SELECT order_id, published_at
FROM gold.fct_orders
ORDER BY order_id
""")
for row in cur.fetchall():
    print(row)

conn.close()

ARJUN SCD2 HISTORY:
('CUST-02', 'Silver', datetime.date(2024, 1, 15), datetime.date(2026, 8, 14))
('CUST-02', 'Gold', datetime.date(2026, 8, 15), datetime.date(9999, 12, 31))

ORD-1002 AFTER:
('ORD-1002', 'Gold', 70, datetime.datetime(2026, 11, 11, 12, 26, 57, 666000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))

GOLD TOTAL AFTER:
(4, 161)

PUBLISHED_AT AFTER:
('ORD-1001', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1002', datetime.datetime(2026, 11, 11, 12, 26, 57, 666000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1003', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1005', datetime.datetime(2026, 11, 11, 6, 56, 47, 42000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))


In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/sql/40_gold_fct_orders.sql")
print(p.read_text())

-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to    

In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt")
for file in p.rglob("*.py"):
    content = file.read_text(errors="ignore")
    if "def quality_checks" in content:
        print("Found in:", file)
        start = content.find("def quality_checks")
        print(content[start:start+5000])

Found in: /content/freshcart_elt/run_pipeline.py
def quality_checks(conn, strict):
    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
    unless --strict is given."""
    import snowflake_io as sf
    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
    checks = [
        ("no fan-out in enrichment (rows out - rows in)",
         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
         lambda v: v == 0, "fail"),
        ("one gold row per order_id (duplicates)",
         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
        ("orders with no customer match",
         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("orders with an amount but no FX rate",
         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
         lambda v: v == 0, "warn"),
        ("orders wit

In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/run_pipeline.py")
lines = p.read_text().splitlines()

for i, line in enumerate(lines, 1):
    if 120 <= i <= 190:
        print(f"{i}: {line}")

120:         run.step("30 customers scd2",  sf.run_sql_file, conn, SQL / "30_dim_customers_scd2.sql", vars_)
121:         run.step("31 fx daily",        sf.run_sql_file, conn, SQL / "31_ref_fx_daily.sql", vars_)
122:         run.step("32 pincode geo",     sf.run_sql_file, conn, SQL / "32_ref_pincode_geo.sql", vars_)
123:         run.step("40 enrich + gold",   sf.run_sql_file, conn, SQL / "40_gold_fct_orders.sql", vars_)
124: 
125:         # ---------------- 4. CHECK ----------------
126:         run.step("quality checks", quality_checks, conn, args.strict)
127:         n, revenue, points = sf.query_one(conn,
128:             "SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
129:         log.info("GOLD: %s orders, revenue ₹%s, %s loyalty points", n, revenue, points)
130:     finally:
131:         conn.close()
132: 
133: 
134: def quality_checks(conn, strict):
135:     """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
136:     unless --

In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/run_pipeline.py")
text = p.read_text()

old = '''        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),'''

new = old + '''
        ("orders with future order_date",
         "SELECT COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup",
         lambda v: v == 0, "fail"),'''

if "orders with future order_date" not in text:
    if old not in text:
        raise ValueError("Could not find the expected location. No changes made.")
    p.write_text(text.replace(old, new, 1))
    print("Sixth quality check added successfully!")
else:
    print("The future-date quality check is already present.")

Sixth quality check added successfully!


In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/mock_data.py")
text = p.read_text()

print(text[:7000])

"""Mock API responses: the 8 FreshCart orders from the course, split by the channel
they came from, plus CRM, FX and pincode responses. Used with --mock so the whole
pipeline runs end to end without any real API.

Note the two order APIs use DIFFERENT JSON shapes. That is normal in real life,
and it is why bronze stores raw JSON and silver unifies it with SQL.
"""

# Web export: flat JSON (rows 1, 3, 4, 5, 6, 8 of the course table)
WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_d

In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/mock_data.py")
text = p.read_text()

mock_order = '''    {"order_id": "ORD-1099", "cust_email": "test@example.com", "cust_name": "Test User",
     "order_date": "2099-01-01", "amount": "500.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2099-01-01 10:00:00"},'''

if '"ORD-1099"' not in text:
    text = text.replace("WEB_ORDERS = [", "WEB_ORDERS = [\n" + mock_order, 1)
    p.write_text(text)
    print("Temporary future-date order added!")
else:
    print("Temporary order is already present.")

Temporary future-date order added!


In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
20:39:55 pipeline  run_id=20260930_203955 run_date=2026-09-30 mock=True
20:39:55 extract   web: 7 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_203955.json.gz
20:39:55 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_203955.json.gz
20:39:55 extract   crm: 5 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_203955.json.gz
20:39:55 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_203955.json.gz
20:39:56 snowflake.connector Python connector starting v5.2.0
20:39:56 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
20:39:56 numexpr.utils NumExpr defaulting to 2 threads.
20:39:56 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
20:39:56 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
20:39:56 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
20:39:5

In [ ]:
from pathlib import Path

p = Path("/content/freshcart_elt/mock_data.py")
text = p.read_text()

start = text.find('    {"order_id": "ORD-1099"')
if start != -1:
    end = text.find('},', start) + 2
    text = text[:start] + text[end:]
    p.write_text(text)
    print("Temporary order ORD-1099 removed successfully!")
else:
    print("Temporary order was already removed.")

Temporary order ORD-1099 removed successfully!


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

cur.execute("""
DELETE FROM BRONZE.ORDERS_RAW
WHERE source = 'web'
AND raw:order_id::STRING = 'ORD-1099'
""")

print("Deleted mock records:", cur.rowcount)

cur.execute("""
SELECT COUNT(*)
FROM BRONZE.ORDERS_RAW
WHERE source = 'web'
AND raw:order_id::STRING = 'ORD-1099'
""")

print("Remaining ORD-1099 records:", cur.fetchone()[0])

cur.close()
conn.close()

Deleted mock records: 1
Remaining ORD-1099 records: 0


## 5. Dry run: extract only, no Snowflake
Writes the raw sample API responses to `landing/`. Then we peek at one raw mobile record.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --dry-run --mock --run-date 2026-09-30
!zcat landing/mobile/*/*.json.gz | head -1

/content/freshcart_elt
14:52:06 pipeline  run_id=20260930_145206 run_date=2026-09-30 mock=True
14:52:06 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_145206.json.gz
14:52:06 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_145206.json.gz
14:52:06 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_145206.json.gz
14:52:06 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_145206.json.gz
14:52:06 pipeline  dry run: landing files written, Snowflake not touched: {'web': PosixPath('/content/freshcart_elt/landing/web/2026-09-30/web__20260930_145206.json.gz'), 'mobile': PosixPath('/content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_145206.json.gz'), 'crm': PosixPath('/content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_145206.json.gz'), 'fx': PosixPath('/content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_1452

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
20:44:03 pipeline  run_id=20260930_204403 run_date=2026-09-30 mock=True
20:44:03 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_204403.json.gz
20:44:03 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_204403.json.gz
20:44:03 extract   crm: 5 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_204403.json.gz
20:44:03 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_204403.json.gz
20:44:03 snowflake.connector Python connector starting v5.2.0
20:44:04 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
20:44:04 numexpr.utils NumExpr defaulting to 2 threads.
20:44:04 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
20:44:04 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
20:44:04 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
20:44:0

In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()
cur.execute("""
SELECT order_id
FROM gold.fct_orders
WHERE order_id = 'ORD-1099'
""")
print(cur.fetchall())

cur.close()
conn.close()

[('ORD-1099',)]


In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse="FRESHCART_WH",
    database="FRESHCART"
)

cur = conn.cursor()

cur.execute("""
DELETE FROM gold.fct_orders
WHERE order_id = 'ORD-1099'
""")

print("Deleted rows:", cur.rowcount)

cur.close()
conn.close()

Deleted rows: 1


## 6. Run the full pipeline on the sample data
`--setup` creates the warehouse, database, schemas and tables. It is only needed the first time; leaving it in is harmless.

Expected last line: **GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points**. One `WARN` for the quarantine rate (2 of 8 rows = 25%) is expected for this sample.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
14:56:30 pipeline  run_id=20260930_145630 run_date=2026-09-30 mock=True
14:56:30 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_145630.json.gz
14:56:30 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_145630.json.gz
14:56:30 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_145630.json.gz
14:56:30 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_145630.json.gz
14:56:30 snowflake.connector Python connector starting v5.2.0
14:56:30 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
14:56:30 numexpr.utils NumExpr defaulting to 2 threads.
14:56:31 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
14:56:31 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
14:56:31 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
14:56:3

## 7. Look inside each layer
`show()` runs a query and returns a table.

In [ ]:
import pandas as pd
conn = sf.connect()

def show(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

show("SELECT ingest_seq, source, raw FROM bronze.orders_raw ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,RAW
0,1,web,"{\n ""amount"": ""1,250.00"",\n ""currency"": ""INR..."
1,2,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
2,3,web,"{\n ""amount"": ""42.50"",\n ""currency"": ""USD"",\..."
3,4,web,"{\n ""amount"": ""-500"",\n ""currency"": ""INR"",\n..."
4,5,web,"{\n ""amount"": ""₹ 899"",\n ""currency"": ""inr"",\..."
5,6,web,"{\n ""amount"": 30,\n ""currency"": ""EUR"",\n ""c..."
6,7,mobile,"{\n ""customer"": {\n ""email"": ""arjun.k@yaho..."
7,8,mobile,"{\n ""customer"": {\n ""email"": ""Meera@Outloo..."


### Silver: cleaned rows (6)

In [ ]:
show("SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, amount_missing FROM silver.orders_clean ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,ORDER_ID,EMAIL,NAME,ORDER_DATE,AMOUNT,CURRENCY,PINCODE,AMOUNT_MISSING
0,1,web,ORD-1001,priya.r@gmail.com,Priya R,2026-09-01,1250.00,INR,600041,False
1,2,web,ORD-1003,meera@outlook.com,Meera S,2026-09-02,None,INR,400050,True
2,3,web,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
3,5,web,ORD-1005,priya.r@gmail.com,Priya R,2026-09-03,899.00,INR,600041,False
4,7,mobile,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
5,8,mobile,ORD-1003,meera@outlook.com,Meera S,2026-09-02,2100.00,INR,400050,False


### Quarantine: rejected rows and why

In [ ]:
show("SELECT run_id, source, order_id, reasons FROM quarantine.orders")

,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20260930_070333,web,ORD-1006,"[\n ""bad_email"",\n ""bad_pincode""\n]"
1,20260930_070333,web,ORD-1004,"[\n ""non_positive_amount""\n]"


### Silver: one row per order (4)

In [ ]:
show("SELECT order_id, ingest_seq, updated_at, amount FROM silver.orders_dedup ORDER BY order_id")

,ORDER_ID,INGEST_SEQ,UPDATED_AT,AMOUNT
0,ORD-1001,1,2026-09-01 10:02:00,1250.00
1,ORD-1002,7,2026-09-01 11:15:00,42.50
2,ORD-1003,8,2026-09-02 18:40:00,2100.00
3,ORD-1005,5,2026-09-03 16:45:00,899.00


### Dim: customer tier history (SCD2)

In [ ]:
show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

,CUST_ID,EMAIL,TIER,VALID_FROM,VALID_TO,IS_CURRENT
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10,2026-09-02,False
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03,9999-12-31,True
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15,9999-12-31,True
3,CUST-03,meera@outlook.com,Platinum,2022-11-01,9999-12-31,True


### Ref: daily FX rates, weekends filled

In [ ]:
show("SELECT * FROM ref.fx_daily WHERE currency = 'USD' ORDER BY rate_date LIMIT 10")

,CURRENCY,RATE_DATE,RATE_TO_INR,IS_CARRIED_FORWARD
0,USD,2026-09-01,83.2000,False
1,USD,2026-09-02,83.3500,False
2,USD,2026-09-03,83.1000,False
3,USD,2026-09-04,83.1000,True
4,USD,2026-09-05,83.1000,True
5,USD,2026-09-06,83.1000,True
6,USD,2026-09-07,83.1000,True
7,USD,2026-09-08,83.1000,True
8,USD,2026-09-09,83.1000,True
9,USD,2026-09-10,83.1000,True


### Ref: pincode geography

In [ ]:
show("SELECT * FROM ref.pincode_geo")

,PINCODE,CITY,STATE,REGION,UPDATED_AT
0,600041,Chennai,Tamil Nadu,South,2026-10-01 00:04:06.456000-07:00
1,560034,Bengaluru,Karnataka,South,2026-10-01 00:04:06.456000-07:00
2,400050,Mumbai,Maharashtra,West,2026-10-01 00:04:06.456000-07:00


### Gold: the final table

In [ ]:
show("SELECT order_id, order_date, tier, amount, currency, fx_rate, amount_inr, loyalty_points, is_high_value, city, region FROM gold.fct_orders ORDER BY order_id")

,ORDER_ID,ORDER_DATE,TIER,AMOUNT,CURRENCY,FX_RATE,AMOUNT_INR,LOYALTY_POINTS,IS_HIGH_VALUE,CITY,REGION
0,ORD-1001,2026-09-01,Silver,1250.00,INR,1.0000,1250.00,12,False,Chennai,South
1,ORD-1002,2026-09-01,Silver,42.50,USD,83.2000,3536.00,35,True,Bengaluru,South
2,ORD-1003,2026-09-02,Platinum,2100.00,INR,1.0000,2100.00,63,True,Mumbai,West
3,ORD-1005,2026-09-03,Gold,899.00,INR,1.0000,899.00,16,False,Chennai,South


### Gold totals

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

,ORDERS,REVENUE_INR,POINTS
0,4,7785.00,126


### Run log

In [ ]:
show("SELECT step, status, detail, logged_at FROM ops.run_log ORDER BY logged_at DESC LIMIT 20")

,STEP,STATUS,DETAIL,LOGGED_AT
0,quality checks,success,2.0s; result='passed',2026-10-01 00:04:12.098000-07:00
1,40 enrich + gold,success,2.1s; result=None,2026-10-01 00:04:09.699000-07:00
2,32 pincode geo,success,1.2s; result=None,2026-10-01 00:04:07.147000-07:00
3,31 fx daily,success,1.4s; result=None,2026-10-01 00:04:04.450000-07:00
4,30 customers scd2,success,1.5s; result=None,2026-10-01 00:04:02.646000-07:00
5,load pincodes,success,2.2s; result=3,2026-10-01 00:04:00.624000-07:00
6,20 deduplicate,success,1.3s; result=None,2026-10-01 00:03:57.597000-07:00
7,10 cleanse orders,success,2.8s; result=None,2026-10-01 00:03:55.823000-07:00
8,load fx,success,1.4s; result=1,2026-10-01 00:03:52.560000-07:00
9,load crm,success,1.6s; result=4,2026-10-01 00:03:50.585000-07:00


## 8. Run it again: nothing should double
A second run extracts the same sample again, so bronze gets new copies. Deduplication and the gold `MERGE` absorb them: the totals stay 4 orders, ₹7,785.00, 126 points.

In [ ]:
!python run_pipeline.py --mock --run-date 2026-09-30
show("SELECT COUNT(*) AS bronze_rows FROM bronze.orders_raw")

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

## 9. Switch to your real APIs
Add these to Colab Secrets, run the cell, then run the pipeline **without** `--mock`.
Exchange rates (frankfurter.app) and pincodes (api.postalpincode.in) are free public APIs and need no key.

If your APIs page their results differently, edit `items_key` / `next_key` in `extract.py`. If their field names differ, edit only the `unified` block at the top of `sql/10_silver_orders_clean.sql`.

In [ ]:
for k in ["WEB_ORDERS_URL", "WEB_ORDERS_TOKEN", "MOBILE_ORDERS_URL", "MOBILE_ORDERS_TOKEN",
          "CRM_TIER_EVENTS_URL", "CRM_TOKEN"]:
    v = secret(k)
    if v:
        os.environ[k] = v
os.environ["FX_URL"] = "https://api.frankfurter.app"
os.environ["FX_BASES"] = "USD"
os.environ["PINCODE_URL"] = "https://api.postalpincode.in/pincode"

# !python run_pipeline.py          # uncomment to run on today's real data

## 10. Clean up (optional)
Removes everything this notebook created in Snowflake. Uncomment to run.

In [ ]:
# conn.cursor().execute("DROP DATABASE IF EXISTS FRESHCART")
# conn.cursor().execute("DROP WAREHOUSE IF EXISTS FRESHCART_WH")
conn.close()